# RNN Time-Series Forecasting with PyTorch
**Dataset:** [Air Pollution Forecasting – LSTM Multivariate](https://www.kaggle.com/datasets/rupakroy/lstm-datasets-multivariate-univariate) (hourly PM2.5 and weather readings)

**Task:** predict future pollution levels from the previous hours of pollution *and* weather.

## 0. Setup and settings

In [ ]:
import glob, json, os, random, time
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# ---------------- settings ----------------
DATA_PATH     = None        # e.g. "pollution.csv"; None = search common locations
TARGET        = "pollution" # column to forecast
HORIZON       = 1           # number of future steps to predict (1 = next hour)
N_TRIALS      = 10          # random-search trials
SEARCH_EPOCHS = 15          # epochs per search trial
FINAL_EPOCHS  = 60          # max epochs for the final models
PATIENCE      = 8           # early-stopping patience for final models
SEED          = 42
OUT_DIR       = "results"
# ----------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Device:", DEVICE)

## 1. Load the data


In [ ]:
if DATA_PATH is None:
    candidates = []
    for root in [".", "/kaggle/input", "/content"]:
        candidates += glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
    if not candidates:
        raise FileNotFoundError("No CSV found. Download the dataset and set DATA_PATH.")
    candidates.sort(key=lambda p: ("pollut" not in os.path.basename(p).lower(), p))
    print("CSV files found:", candidates)
    DATA_PATH = candidates[0]
print("Using:", DATA_PATH)

raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.head()

## 2. Preprocessing

| Step | Why |
|---|---|
| Parse dates → time index | Needed for chronological order and calendar features |
| Interpolate missing numeric values | RNNs cannot take NaNs; interpolation keeps the series continuous |
| One-hot encode categorical columns (wind direction) | Networks need numeric input |
| Add sine/cosine of hour and month | Gives the model smooth daily and seasonal cycles |
| **Chronological** 70/15/15 split | Random splitting would leak future information into training |
| Min-max scaling **fitted on train only** | Keeps validation and test statistics out of training |
| Sliding windows | Each sample is `LOOKBACK` past steps of all features, label is the next `HORIZON` target values |

In [ ]:
def load_and_clean(df, target):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    target = target.lower()

    date_col = "date" if "date" in df.columns else df.columns[0]
    parsed = pd.to_datetime(df[date_col], errors="coerce")
    if parsed.notna().mean() > 0.9:
        df = df.drop(columns=[date_col]).set_index(parsed.rename("date"))
        df = df[~df.index.isna()].sort_index()
    else:
        print(f"[warn] could not parse '{date_col}' as dates; using row order.")
        df = df.reset_index(drop=True)

    if target not in df.columns:
        raise ValueError(f"target '{target}' not in columns {list(df.columns)}")

    cat_cols = [c for c in df.columns if df[c].dtype == object]
    num_cols = [c for c in df.columns if c not in cat_cols]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    print("Missing values before cleaning:\n", df.isna().sum().to_string(), "\n")
    df[num_cols] = df[num_cols].interpolate(limit_direction="both")
    for c in cat_cols:
        df[c] = df[c].ffill().bfill()
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, dtype=float)

    if isinstance(df.index, pd.DatetimeIndex):
        h, m = df.index.hour, df.index.month
        df["hour_sin"], df["hour_cos"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
        df["month_sin"], df["month_cos"] = np.sin(2*np.pi*m/12), np.cos(2*np.pi*m/12)

    cols = [target] + [c for c in df.columns if c != target]   # target first
    return df[cols].astype(np.float32)


df = load_and_clean(raw, TARGET)
print("Clean shape:", df.shape)
print("Features:", list(df.columns))
df.describe().T[["mean", "std", "min", "max"]].round(2)

### Exploratory plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df[TARGET.lower()].plot(ax=ax[0], lw=.5, title=f"{TARGET} over time")
base_feats = [c for c in df.columns if not c.endswith(("_sin", "_cos")) and "_" not in c or c in ("wnd_spd",)]
corr = df[base_feats].corr()[TARGET.lower()].drop(TARGET.lower()).sort_values()
corr.plot.barh(ax=ax[1], title=f"Correlation with {TARGET}")
plt.tight_layout(); plt.show()

In [ ]:
def make_windows(arr, lookback, horizon):
    # X[i] = arr[i:i+lookback]  ;  y[i] = next `horizon` target values (column 0)
    n = len(arr) - lookback - horizon + 1
    idx = np.arange(lookback)[None, :] + np.arange(n)[:, None]
    yidx = lookback + np.arange(horizon)[None, :] + np.arange(n)[:, None]
    return arr[idx].astype(np.float32), arr[yidx, 0].astype(np.float32)


@dataclass
class Data:
    train: tuple
    val: tuple
    test: tuple
    scaler: MinMaxScaler
    test_index: pd.Index


def prepare_data(df, lookback, horizon, train_frac=0.70, val_frac=0.15):
    values = df.values
    T = len(values)
    t_end, v_end = int(T*train_frac), int(T*(train_frac+val_frac))
    scaler = MinMaxScaler().fit(values[:t_end])          # train only -> no leakage
    scaled = scaler.transform(values).astype(np.float32)
    tr = make_windows(scaled[:t_end], lookback, horizon)
    va = make_windows(scaled[t_end-lookback:v_end], lookback, horizon)   # borrow history
    te = make_windows(scaled[v_end-lookback:], lookback, horizon)
    return Data(tr, va, te, scaler, df.index[v_end:][:len(te[0])])


def inverse_target(scaler, y):
    return (y - scaler.min_[0]) / scaler.scale_[0]


LOOKBACKS = [12, 24, 48]
data_by_lookback = {lb: prepare_data(df, lb, HORIZON) for lb in LOOKBACKS}
d = data_by_lookback[24]
print(f"Lookback 24 -> X shape {d.train[0].shape}; "
      f"train/val/test windows: {len(d.train[0])}/{len(d.val[0])}/{len(d.test[0])}")

## 3. Model

- **Vanilla RNN:** simplest, but prone to vanishing gradients on long windows.
- **GRU:** gated, fewer parameters than LSTM.
- **LSTM:** gated with a separate cell state; usually the strongest on longer dependencies.

In [ ]:
class RNNForecaster(nn.Module):
    def __init__(self, n_features, hidden, layers, dropout, cell="lstm", horizon=1):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[cell]
        extra = {"nonlinearity": "tanh"} if cell == "rnn" else {}
        self.rnn = rnn_cls(n_features, hidden, num_layers=layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0, **extra)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(dropout),
                                  nn.Linear(hidden, hidden // 2), nn.ReLU(),
                                  nn.Linear(hidden // 2, horizon))

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])


@dataclass
class Config:
    cell: str = "lstm"
    hidden: int = 64
    layers: int = 2
    dropout: float = 0.2
    lr: float = 1e-3
    batch_size: int = 128
    lookback: int = 24
    weight_decay: float = 1e-5

## 4. Training
- Loss: MSE on scaled data
- Optimiser: Adam with weight decay
- Gradient-norm clipping at 1.0 (stabilises recurrent training)
- `ReduceLROnPlateau` halves the learning rate when validation loss stalls
- Early stopping restores the best-validation weights

In [ ]:
def make_loader(split, batch_size, shuffle):
    X, y = (torch.from_numpy(a) for a in split)
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def predict(model, split, bs=512):
    model.eval()
    X = torch.from_numpy(split[0])
    return torch.cat([model(X[i:i+bs].to(DEVICE)).cpu() for i in range(0, len(X), bs)]).numpy()


def train_model(cfg, epochs, patience, seed=SEED, verbose=True):
    set_seed(seed)
    data = data_by_lookback[cfg.lookback]
    model = RNNForecaster(data.train[0].shape[2], cfg.hidden, cfg.layers, cfg.dropout,
                          cfg.cell, HORIZON).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    loss_fn = nn.MSELoss()
    tr_loader = make_loader(data.train, cfg.batch_size, True)
    Xv, yv = (torch.from_numpy(a).to(DEVICE) for a in data.val)

    best, best_state, bad = float("inf"), None, 0
    hist = {"train": [], "val": []}
    for ep in range(1, epochs + 1):
        model.train(); tot = n = 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(xb); n += len(xb)
        model.eval()
        with torch.no_grad():
            val = sum(loss_fn(model(Xv[i:i+1024]), yv[i:i+1024]).item() * len(Xv[i:i+1024])
                      for i in range(0, len(Xv), 1024)) / len(Xv)
        hist["train"].append(tot / n); hist["val"].append(val)
        sched.step(val)
        if verbose and (ep == 1 or ep % 5 == 0):
            print(f"  epoch {ep:3d} | train {tot/n:.5f} | val {val:.5f} | lr {opt.param_groups[0]['lr']:.1e}")
        if val < best - 1e-6:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                if verbose: print(f"  early stop at epoch {ep}")
                break
    model.load_state_dict(best_state)
    return model, hist, best

## 5. Hyperparameter search
Random search over cell type, width, depth, dropout, learning rate, batch size and lookback window. Configurations are ranked by **validation** MSE; the test set is not touched here.

In [ ]:
search_space = {
    "cell":       ["rnn", "gru", "lstm"],
    "hidden":     [32, 64, 128],
    "layers":     [1, 2, 3],
    "dropout":    [0.0, 0.2, 0.3],
    "lr":         [3e-3, 1e-3, 5e-4],
    "batch_size": [64, 128, 256],
    "lookback":   LOOKBACKS,
}
rng = random.Random(SEED)
trials = [Config()]                                   # default config always included
while len(trials) < N_TRIALS:
    c = Config(**{k: rng.choice(v) for k, v in search_space.items()})
    if c not in trials:
        trials.append(c)

rows = []
for i, cfg in enumerate(trials, 1):
    t0 = time.time()
    print(f"[{i}/{len(trials)}] {asdict(cfg)}")
    _, _, best_val = train_model(cfg, SEARCH_EPOCHS, patience=4, verbose=False)
    print(f"    val MSE (scaled) = {best_val:.5f}   ({time.time()-t0:.0f}s)")
    rows.append({**asdict(cfg), "val_mse": best_val})

search_df = pd.DataFrame(rows).sort_values("val_mse").reset_index(drop=True)
search_df.to_csv(f"{OUT_DIR}/hyperparameter_search.csv", index=False)
search_df

## 6. Final training and architecture comparison
The best configuration is retrained for longer with each of the three cell types so the architectures can be compared fairly under identical hyperparameters. The final model is selected by **validation** loss.

In [ ]:
best_row = search_df.loc[0]
best_cfg = Config(**{k: type(getattr(Config(), k))(best_row[k]) for k in asdict(Config())})
print("Best config from search:", asdict(best_cfg))


def metrics(y_true, y_pred):
    err = y_pred - y_true
    return {"RMSE": float(np.sqrt(np.mean(err**2))),
            "MAE": float(np.mean(np.abs(err))),
            "R2": float(1 - np.sum(err**2) / np.sum((y_true - y_true.mean())**2))}


def evaluate(model, data, split="test"):
    s = getattr(data, split)
    pred = inverse_target(data.scaler, predict(model, s))
    true = inverse_target(data.scaler, s[1])
    last = inverse_target(data.scaler, s[0][:, -1, 0])[:, None]        # persistence baseline
    base = np.repeat(last, true.shape[1], axis=1)
    return pred, true, base, metrics(true, pred), metrics(true, base)


runs, results = {}, {}
for cell in ["rnn", "gru", "lstm"]:
    cfg = Config(**{**asdict(best_cfg), "cell": cell})
    print(f"\n=== {cell.upper()} ===")
    model, hist, best_val = train_model(cfg, FINAL_EPOCHS, PATIENCE)
    data = data_by_lookback[cfg.lookback]
    pred, true, base, m_model, m_base = evaluate(model, data)
    runs[cell] = dict(model=model, hist=hist, pred=pred, true=true, base=base, data=data)
    results[cell] = {"val_mse_scaled": best_val, "test": m_model,
                     "params": sum(p.numel() for p in model.parameters()),
                     "epochs_run": len(hist["train"])}
    results["persistence_baseline"] = {"test": m_base}

winner = min(["rnn", "gru", "lstm"], key=lambda c: results[c]["val_mse_scaled"])
results["selected_model"] = winner
results["selected_config"] = asdict(Config(**{**asdict(best_cfg), "cell": winner}))
results["horizon"] = HORIZON
print("\nSelected by validation loss:", winner.upper())

## 7. Evaluation on the test set

In [ ]:
table = pd.DataFrame({n: results[n]["test"] for n in ["persistence_baseline", "rnn", "gru", "lstm"]}).T
table["params"] = [None] + [results[c]["params"] for c in ["rnn", "gru", "lstm"]]
table["val_mse_scaled"] = [None] + [results[c]["val_mse_scaled"] for c in ["rnn", "gru", "lstm"]]
print("Metrics in original units (lower RMSE/MAE and higher R2 are better)")
table.round(4)

In [ ]:
w = runs[winner]
fig, ax = plt.subplots(2, 2, figsize=(15, 9))

for c, col in zip(["rnn", "gru", "lstm"], ["tab:green", "tab:orange", "tab:blue"]):
    ax[0, 0].plot(runs[c]["hist"]["val"], label=f"{c.upper()} val", color=col)
    ax[0, 0].plot(runs[c]["hist"]["train"], ls="--", color=col, alpha=.6, label=f"{c.upper()} train")
ax[0, 0].set(title="Training curves (MSE, scaled)", xlabel="epoch"); ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)

n = 300
x = w["data"].test_index[:n]
ax[0, 1].plot(x, w["true"][:n, 0], label="actual", lw=1.5)
ax[0, 1].plot(x, w["pred"][:n, 0], label=f"{winner.upper()} forecast", lw=1.2)
ax[0, 1].plot(x, w["base"][:n, 0], label="persistence", lw=.8, ls="--", alpha=.7)
ax[0, 1].set_title(f"Test set, first {n} steps"); ax[0, 1].legend(); ax[0, 1].grid(alpha=.3)
ax[0, 1].tick_params(axis="x", rotation=30)

ax[1, 0].scatter(w["true"][:, 0], w["pred"][:, 0], s=4, alpha=.3)
lim = [0, max(w["true"].max(), w["pred"].max())]
ax[1, 0].plot(lim, lim, "r--", lw=1)
ax[1, 0].set(title="Predicted vs actual (test)", xlabel="actual", ylabel="predicted")

resid = w["pred"][:, 0] - w["true"][:, 0]
ax[1, 1].hist(resid, bins=60)
ax[1, 1].set(title=f"Residuals (mean {resid.mean():.2f}, std {resid.std():.2f})", xlabel="prediction error")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/evaluation.png", dpi=150)
plt.show()

In [ ]:
if HORIZON > 1:
    per_step = [float(np.sqrt(np.mean((w["pred"][:, k] - w["true"][:, k])**2))) for k in range(HORIZON)]
    results["per_step_RMSE"] = per_step
    plt.figure(figsize=(6, 3.5))
    plt.plot(range(1, HORIZON + 1), per_step, marker="o")
    plt.xlabel("steps ahead"); plt.ylabel("RMSE"); plt.title("Error vs forecast distance"); plt.grid(alpha=.3)
    plt.show()

torch.save(w["model"].state_dict(), f"{OUT_DIR}/best_model_{winner}.pt")
with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved model, metrics.json, hyperparameter_search.csv and evaluation.png to", OUT_DIR)